
# Decision Tree

- **不使用 scikit-learn / pandas / matplotlib**，只用 `numpy`。  
- 支持连续特征与多分类；含训练、预测、精度评估与简单超参。  
- 示例数据使用随机可分数据集（也可替换为你自己的 `X, y`）。  
- 更新时间：2025-10-17 21:48


## 1. 依赖与环境

In [1]:

import sys, numpy as np
np.set_printoptions(suppress=True, precision=4)
print("Python:", sys.version.split()[0])
print("Numpy :", np.__version__)


Python: 3.14.0
Numpy : 2.3.4


## 2. 生成一个玩具数据集（可替换为你自己的 X, y）

In [4]:

rng = np.random.default_rng(42)

def make_blobs(n_samples=300, centers=3, n_features=2, cluster_std=1.2, center_box=(-5, 5), seed=42):
    rng = np.random.default_rng(seed)
    centers_pts = rng.uniform(center_box[0], center_box[1], size=(centers, n_features))
    X = []
    y = []
    per = n_samples // centers
    for i in range(centers):
        cov = np.eye(n_features) * (cluster_std ** 2)
        pts = rng.multivariate_normal(centers_pts[i], cov, size=per)
        X.append(pts); y.append(np.full(per, i, dtype=int))
    X = np.vstack(X); y = np.concatenate(y)
    # 若不能整除，补齐剩余样本
    rest = n_samples - len(y)
    if rest > 0:
        extra = rng.multivariate_normal(centers_pts[0], np.eye(n_features)*(cluster_std**2), size=rest)
        X = np.vstack([X, extra]); y = np.concatenate([y, np.zeros(rest, dtype=int)])
    return X, y

X, y = make_blobs(n_samples=300, centers=3, n_features=2, cluster_std=1.1, seed=7)
print("X shape:", X.shape, " y shape:", y.shape, "classes:", np.unique(y))


X shape: (300, 2)  y shape: (300,) classes: [0 1 2]


## 3. 划分训练/测试集

In [5]:

def train_test_split_np(X, y, test_size=0.2, seed=0, stratify=True):
    rng = np.random.default_rng(seed)
    n = len(y)
    idx = np.arange(n)
    if stratify:
        # 分层抽样
        all_idx = []
        for c in np.unique(y):
            c_idx = idx[y == c]
            rng.shuffle(c_idx)
            split = int(len(c_idx) * (1 - test_size))
            all_idx.append((c_idx[:split], c_idx[split:]))
        train_idx = np.concatenate([a for a,_ in all_idx])
        test_idx  = np.concatenate([b for _,b in all_idx])
    else:
        rng.shuffle(idx)
        split = int(n * (1 - test_size))
        train_idx, test_idx = idx[:split], idx[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_np(X, y, test_size=0.25, seed=42, stratify=True)
X_train.shape, X_test.shape


((225, 2), (75, 2))

## 4. 决策树实现（CART，基尼不纯度）

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Node:
    is_leaf: bool
    pred: int = None
    feature: int = None
    threshold: float = None
    left: 'Node' = None
    right: 'Node' = None
    depth: int = 0
    n_samples: int = 0

# ---------- 1) 用熵 ----------
def entropy(y, eps=1e-12):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + eps))  # 信息熵

def majority_class(y):
    classes, counts = np.unique(y, return_counts=True)
    return classes[np.argmax(counts)]

# ---------- 2) 用信息增益（可选：信息增益比） ----------
def best_split_cart_info_gain(X, y, min_samples_leaf=1, use_gain_ratio=False, eps=1e-12):
    # 返回 (best_feature, best_threshold, best_weighted_child_entropy, best_left_mask, best_right_mask)
    n, d = X.shape
    base_ent = entropy(y)
    best_gain = 0.0
    best = (None, None, None, None, None)

    for j in range(d):
        order = np.argsort(X[:, j])
        xj = X[order, j]
        # y_sorted 未直接使用，这里保留排序以便只试相邻不同阈值
        # y_sorted = y[order]

        for i in range(1, n):
            if xj[i] == xj[i-1]:
                continue
            th = (xj[i] + xj[i-1]) / 2.0

            left_mask = X[:, j] <= th
            right_mask = ~left_mask
            nL, nR = left_mask.sum(), right_mask.sum()
            if nL < min_samples_leaf or nR < min_samples_leaf:
                continue

            ent_L = entropy(y[left_mask])
            ent_R = entropy(y[right_mask])
            # weighted average by their feature numbers/total features
            ent_weighted = (nL * ent_L + nR * ent_R) / n

            # 信息增益
            gain = base_ent - ent_weighted
            # gain = base_ent-(ent_L + ent_R)

            if use_gain_ratio:
                # Split Info（固有值）: - Σ (|S_i|/|S|) log2(|S_i|/|S|)
                pL, pR = nL / n, nR / n
                split_info = -(pL * np.log2(pL + eps) + pR * np.log2(pR + eps))
                if split_info > eps:
                    gain = gain / split_info
                else:
                    # 极端不平衡时，避免除零：跳过或保持原 gain 均可
                    continue

            if gain > best_gain:
                best_gain = gain
                best = (j, th, ent_weighted, left_mask, right_mask)

    return best, best_gain

class DecisionTreeClassifierNP:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 max_features=None, random_state=None, use_gain_ratio=False):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.root = None
        self.use_gain_ratio = use_gain_ratio  # True 则用信息增益比

    def fit(self, X, y):
        self.n_classes_ = len(np.unique(y))
        self.root = self._build(X, y, depth=0)
        return self

    def _build(self, X, y, depth):
        node = Node(is_leaf=False, depth=depth, n_samples=len(y))

        # 停止条件
        if (self.max_depth is not None and depth >= self.max_depth) \
           or len(np.unique(y)) == 1 \
           or len(y) < self.min_samples_split:
            node.is_leaf = True
            node.pred = majority_class(y)
            return node

        (feat, thr, ent_w, left_mask, right_mask), gain = best_split_cart_info_gain(
            X, y,
            min_samples_leaf=self.min_samples_leaf,
            use_gain_ratio=self.use_gain_ratio
        )

        if feat is None or gain <= 1e-12:
            node.is_leaf = True
            node.pred = majority_class(y)
            return node

        node.feature = feat
        node.threshold = thr
        node.left = self._build(X[left_mask], y[left_mask], depth+1)
        node.right = self._build(X[right_mask], y[right_mask], depth+1)
        return node

    def _predict_one(self, x, node):
        while not node.is_leaf:
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.pred

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X], dtype=int)

    def score(self, X, y):
        y_pred = self.predict(X)
        return (y_pred == y).mean()


## 5. 评估：训练/测试精度

In [15]:

train_acc = tree.score(X_train, y_train)
test_acc  = tree.score(X_test,  y_test)
print(f"Train acc: {train_acc:.4f}")
print(f"Test  acc: {test_acc:.4f}")


Train acc: 0.9911
Test  acc: 0.9600


## 6. 简单 K 折交叉验证（仅使用 numpy）

In [8]:

def k_fold_indices(n_samples, k=5, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(n_samples)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    return folds

def cross_val_score_np(model_cls, X, y, k=5, seed=0, **model_kwargs):
    folds = k_fold_indices(len(y), k=k, seed=seed)
    scores = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        m = model_cls(**model_kwargs)
        m.fit(X[train_idx], y[train_idx])
        scores.append(m.score(X[test_idx], y[test_idx]))
    return np.array(scores)

cv_scores = cross_val_score_np(DecisionTreeClassifierNP, X, y, k=5, seed=42, max_depth=6)
print("CV scores:", cv_scores)
print("Mean:", cv_scores.mean(), "Std:", cv_scores.std())


CV scores: [0.9        0.88333333 0.98333333 0.93333333 0.9       ]
Mean: 0.9200000000000002 Std: 0.03559026084010435


## 7. 简单网格搜索（纯手写）

In [9]:

def grid_search_dt(X, y, param_grid, k=5, seed=0):
    keys = list(param_grid.keys())
    best_params = None
    best_score = -1
    results = []
    # 笛卡尔积
    def rec(idx, cur):
        nonlocal best_params, best_score, results
        if idx == len(keys):
            params = dict(cur)
            scores = cross_val_score_np(DecisionTreeClassifierNP, X, y, k=k, seed=seed, **params)
            mean = scores.mean()
            results.append((params, mean))
            if mean > best_score:
                best_score = mean
                best_params = params
            return
        key = keys[idx]
        for v in param_grid[key]:
            cur[key] = v
            rec(idx+1, cur)
        cur.pop(key, None)

    rec(0, {})
    return best_params, best_score, results

param_grid = {
    "max_depth": [None, 3, 4, 5, 6, 8],
    "min_samples_split": [2, 4, 8],
    "min_samples_leaf": [1, 2, 4]
}

best_params, best_score, results = grid_search_dt(X, y, param_grid, k=4, seed=1)
print("Best params:", best_params)
print("Best CV mean acc:", round(float(best_score), 4))


Best params: {'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 4}
Best CV mean acc: 0.9467


## 8. 用最优超参重训并在测试集评估

In [ ]:

best_tree = DecisionTreeClassifierNP(**best_params)
best_tree.fit(X_train, y_train)
print("Train acc:", round(float(best_tree.score(X_train, y_train)), 4))
print("Test  acc:",  round(float(best_tree.score(X_test,  y_test)), 4))


## 9. 替换为你的数据

In [ ]:

# 假设你有自定义数据：X_custom (n_samples, n_features), y_custom (n_samples,)
# 只需：
# tree = DecisionTreeClassifierNP(max_depth=6, min_samples_split=2, min_samples_leaf=1)
# tree.fit(X_custom, y_custom)
# y_pred = tree.predict(X_custom)
# print('acc:', (y_pred == y_custom).mean())
pass



## 10. 说明与可扩展点
- 本实现为 **CART 分类树**：二叉划分，使用 **Gini** 作为不纯度指标。  
- 连续特征阈值：对每个特征排序，在相邻不同值中点上尝试。  
- 早停条件：`max_depth` / `min_samples_split` / `min_samples_leaf`。  
- 可扩展：
  - 支持 **信息增益/熵** 替代 Gini；
  - 加入 **剪枝**（代价复杂度剪枝 / 验证集剪枝）；
  - 支持 **随机子特征**（配合 bagging = 随机森林）；
  - 导出为可读的规则字符串；
  - 加速：针对大数据可用直方图近似搜索阈值。
